# Industry Lab: Customer Churn — Logistic Regression vs k-NN

**Scaler CML | Applied ML (Intro)**  
**Instructor:** Chitwan Manchanda  
**Audience:** Group B (through Session 9 concepts)  
**Format:** ~90 min live demo + light student coding (TODO cells)

### Story
You are an ML engineer shipping a **customer churn** model for a SaaS / telco-style product company.  
Goal: predict who will churn next month so retention can intervene.

### Scope (allowed today)
- Logistic regression + **k-NN**
- Train/test split, **cross-validation**, regularization via `C`, bias–variance via `C` and `k`
- Metrics: accuracy, precision, recall, F1, ROC-AUC, confusion matrix
- Class imbalance intuition + **threshold tuning**
- **No** trees, random forests, boosting, SVM, or neural nets


# Lab agenda (~90 min)

1. **Business problem** → binary label, FP vs FN costs (~8 min)
2. **Data + EDA** + split discipline (~10 min)
3. **Implement logistic regression** end-to-end (~15 min)
4. **Bias–variance / capacity** via `C`; CV to choose `C` (~12 min)
5. **Implement k-NN** end-to-end; CV to choose `k` (~15 min)
6. **Compare** models: CV table, confusion matrices, when to prefer which (~15 min)
7. **Threshold tuning** for the business objective (~10 min)
8. **Production checklist** + wrap (~5 min)


# 0. Setup

Run this cell first. Offline-friendly: we build a **synthetic churn-like** tabular dataset with `make_classification` and industry-style column names.


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, cross_validate
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, classification_report, precision_recall_curve
)

np.random.seed(42)
pd.set_option("display.precision", 3)
print("Ready.")


# 1. Business problem → label & costs

**Product:** monthly SaaS / telco subscription.  
**Label:** `churned` ∈ {0, 1} — did the customer leave in the next billing cycle?

| Error | Meaning | Typical business cost |
|---|---|---|
| **False Positive (FP)** | Predicted churn, but customer stays | Waste a retention offer / coupon / CSM time |
| **False Negative (FN)** | Missed a true churner | Lost revenue + acquisition cost to replace them |

In many retention setups **FN is more expensive** than FP → we often favour **recall** (or a cost-weighted threshold), not raw accuracy.

**Default operating point** `0.5` is rarely the product optimum. We will tune the threshold on validation later.


# 2. Data: synthetic industry churn table

We invent realistic feature names. Mild class imbalance (~18–22% churn) so accuracy alone is misleading.


In [ ]:
FEATURE_NAMES = [
    "months_as_customer",
    "monthly_charges",
    "total_usage_gb",
    "support_tickets_90d",
    "late_payments_12m",
    "contract_months_left",
    "addon_count",
    "login_days_30d",
]

X_raw, y = make_classification(
    n_samples=4000,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_repeated=0,
    n_classes=2,
    weights=[0.80, 0.20],   # ~20% churn — mild imbalance
    class_sep=1.1,
    flip_y=0.03,
    random_state=42,
)

# Rescale columns to look like industry units (cosmetic, before StandardScaler later)
scales = np.array([36, 80, 200, 5, 2, 12, 3, 20], dtype=float)
shifts = np.array([12, 40, 50, 0, 0, 3, 1, 5], dtype=float)
X = X_raw * scales + shifts

df = pd.DataFrame(X, columns=FEATURE_NAMES)
df["churned"] = y

print(df.shape)
print("Churn rate:", round(df["churned"].mean(), 3))
df.head()


## Quick EDA (live)
Look at class balance and a couple of feature summaries. Keep it light — this is not a full EDA course.


In [ ]:
print(df["churned"].value_counts(normalize=True).rename({0: "stay", 1: "churn"}))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
df["churned"].value_counts().plot(kind="bar", ax=axes[0], color=["#4C78A8", "#E45756"])
axes[0].set_title("Class counts")
axes[0].set_xticklabels(["stay (0)", "churn (1)"], rotation=0)

df.boxplot(column="monthly_charges", by="churned", ax=axes[1])
axes[1].set_title("monthly_charges by churn")
axes[1].set_xlabel("churned")
plt.suptitle("")
plt.tight_layout()
plt.show()

df.groupby("churned")[FEATURE_NAMES].mean().T.round(2)


## Split discipline: train / validation / test

**Rule:** the **test** set is a sealed final report card.  
We tune hyperparameters (`C`, `k`) and thresholds on **train** via **cross-validation** (or a dedicated validation fold) — **do not peek at test** while choosing them.


In [ ]:
X = df[FEATURE_NAMES]
y = df["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print("Train:", X_train.shape, "churn rate:", round(y_train.mean(), 3))
print("Test: ", X_test.shape, "churn rate:", round(y_test.mean(), 3))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# 3. Implement Logistic Regression (full pipeline)

**Pattern:** `StandardScaler` → `LogisticRegression` → `predict_proba` → metrics → (later) threshold.

`C` in sklearn is **inverse** regularization strength: **small C → strong L2 → simpler / higher bias**; **large C → weaker penalty → more capacity / higher variance**.


In [ ]:
# --- Full logistic implementation (baseline C=1.0) ---
log_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        C=1.0,
        solver="lbfgs",
        max_iter=2000,
        random_state=42,
    )),
])

log_pipe.fit(X_train, y_train)

# Probabilities for the positive class (churn = 1)
y_prob_log = log_pipe.predict_proba(X_test)[:, 1]
y_pred_log = (y_prob_log >= 0.5).astype(int)   # default threshold

def report_metrics(y_true, y_pred, y_prob, title="Model"):
    print(f"=== {title} ===")
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.3f}")
    print(f"Precision: {precision_score(y_true, y_pred):.3f}")
    print(f"Recall   : {recall_score(y_true, y_pred):.3f}")
    print(f"F1       : {f1_score(y_true, y_pred):.3f}")
    print(f"ROC-AUC  : {roc_auc_score(y_true, y_prob):.3f}")
    print()
    print(classification_report(y_true, y_pred, target_names=["stay", "churn"]))

report_metrics(y_test, y_pred_log, y_prob_log, "Logistic (C=1.0, thr=0.5) — TEST peek for demo only")

fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_log, display_labels=["stay", "churn"], ax=ax)
ax.set_title("Logistic — confusion matrix (test, thr=0.5)")
plt.tight_layout()
plt.show()


### Interpret coefficients (why product teams like logistic)

After scaling, each coefficient ≈ change in **log-odds** of churn for a +1σ move in that feature.


In [ ]:
coefs = pd.Series(log_pipe.named_steps["clf"].coef_.ravel(), index=FEATURE_NAMES)
coefs_sorted = coefs.sort_values()
ax = coefs_sorted.plot(kind="barh", figsize=(7, 4), color="#4C78A8")
ax.set_xlabel("Coefficient (after StandardScaler)")
ax.set_title("Logistic coefficients — direction of association with churn")
plt.tight_layout()
plt.show()
coefs_sorted.round(3)


### TODO 1 (students) — try a different `C`

Change `C` to something small (e.g. `0.01`) and something large (e.g. `100`).  
Refit on **train**, report **train** and **test** F1. What happens to the train–test gap?


In [ ]:
# TODO 1: fill in C_try and complete the loop
for C_try in [0.01, 1.0, 100.0]:  # <-- try your own values too
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(C=C_try, max_iter=2000, random_state=42)),
    ])
    pipe.fit(X_train, y_train)
    # TODO: compute train_f1 and test_f1 using f1_score
    train_pred = pipe.predict(X_train)
    test_pred = pipe.predict(X_test)
    train_f1 = f1_score(y_train, train_pred)
    test_f1 = f1_score(y_test, test_pred)
    print(f"C={C_try:7.2f}  train F1={train_f1:.3f}  test F1={test_f1:.3f}  gap={train_f1 - test_f1:.3f}")


# 4. Cross-validation to choose logistic `C`

We score with **F1** (positive = churn) on stratified 5-fold CV over **train only**. Then lock the best `C` and only then evaluate on test.


In [ ]:
C_grid = [0.01, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0]
cv_rows = []

for C in C_grid:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(C=C, max_iter=2000, random_state=42)),
    ])
    scores = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring={"f1": "f1", "roc_auc": "roc_auc"},
        n_jobs=-1,
    )
    cv_rows.append({
        "model": "logistic",
        "param": f"C={C}",
        "cv_f1_mean": scores["test_f1"].mean(),
        "cv_f1_std": scores["test_f1"].std(),
        "cv_auc_mean": scores["test_roc_auc"].mean(),
        "cv_auc_std": scores["test_roc_auc"].std(),
        "C": C,
    })

log_cv = pd.DataFrame(cv_rows)
display(log_cv[["param", "cv_f1_mean", "cv_f1_std", "cv_auc_mean", "cv_auc_std"]].round(3))

best_C = log_cv.loc[log_cv["cv_f1_mean"].idxmax(), "C"]
print("Best C by CV F1:", best_C)

# Refit best logistic on full train
best_log = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(C=best_C, max_iter=2000, random_state=42)),
])
best_log.fit(X_train, y_train)
y_prob_best_log = best_log.predict_proba(X_test)[:, 1]
y_pred_best_log = (y_prob_best_log >= 0.5).astype(int)
report_metrics(y_test, y_pred_best_log, y_prob_best_log, f"Best Logistic (C={best_C}, thr=0.5)")


# 5. Implement k-NN (full pipeline)

**Pattern:** `StandardScaler` (critical for distance) → choose `k` → `KNeighborsClassifier` → `predict` / `predict_proba` → metrics.

Bias–variance knob:
- **Small k** → wiggly decision regions → **high variance** (can overfit noise)
- **Large k** → smoother → **higher bias** (can underfit local structure)


In [ ]:
# --- Full k-NN implementation (baseline k=11) ---
knn_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNeighborsClassifier(n_neighbors=11, weights="uniform", metric="minkowski", p=2)),
])

knn_pipe.fit(X_train, y_train)

y_prob_knn = knn_pipe.predict_proba(X_test)[:, 1]
y_pred_knn = knn_pipe.predict(X_test)   # equivalent to majority vote among k neighbours

report_metrics(y_test, y_pred_knn, y_prob_knn, "k-NN (k=11, thr≈0.5 via votes) — TEST peek for demo")

fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_knn, display_labels=["stay", "churn"], ax=ax)
ax.set_title("k-NN — confusion matrix (test, k=11)")
plt.tight_layout()
plt.show()


### TODO 2 (students) — why scaling matters for k-NN

Fit k-NN **without** a scaler (raw features) vs **with** `StandardScaler`. Compare test F1.  
(Hint: `monthly_charges` and `total_usage_gb` dwarf `late_payments_12m` in raw units.)


In [ ]:
# TODO 2
k = 11
knn_raw = KNeighborsClassifier(n_neighbors=k)
knn_raw.fit(X_train, y_train)
f1_raw = f1_score(y_test, knn_raw.predict(X_test))

knn_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNeighborsClassifier(n_neighbors=k)),
])
knn_scaled.fit(X_train, y_train)
f1_scaled = f1_score(y_test, knn_scaled.predict(X_test))

print(f"k-NN k={k}  WITHOUT scaler  test F1={f1_raw:.3f}")
print(f"k-NN k={k}  WITH scaler     test F1={f1_scaled:.3f}")


### Cross-validation to choose `k`


In [ ]:
k_grid = [1, 3, 5, 7, 11, 15, 21, 31, 51]
knn_rows = []

for k in k_grid:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=k)),
    ])
    scores = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring={"f1": "f1", "roc_auc": "roc_auc"},
        n_jobs=-1,
    )
    knn_rows.append({
        "model": "knn",
        "param": f"k={k}",
        "cv_f1_mean": scores["test_f1"].mean(),
        "cv_f1_std": scores["test_f1"].std(),
        "cv_auc_mean": scores["test_roc_auc"].mean(),
        "cv_auc_std": scores["test_roc_auc"].std(),
        "k": k,
    })

knn_cv = pd.DataFrame(knn_rows)
display(knn_cv[["param", "cv_f1_mean", "cv_f1_std", "cv_auc_mean", "cv_auc_std"]].round(3))

best_k = int(knn_cv.loc[knn_cv["cv_f1_mean"].idxmax(), "k"])
print("Best k by CV F1:", best_k)

# Optional: plot CV F1 vs k (bias–variance story)
plt.figure(figsize=(7, 3.5))
plt.errorbar(knn_cv["k"], knn_cv["cv_f1_mean"], yerr=knn_cv["cv_f1_std"], marker="o", color="#E45756")
plt.xlabel("k")
plt.ylabel("CV F1 (mean ± std)")
plt.title("k-NN: CV F1 vs k (small k → higher variance)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNeighborsClassifier(n_neighbors=best_k)),
])
best_knn.fit(X_train, y_train)
y_prob_best_knn = best_knn.predict_proba(X_test)[:, 1]
y_pred_best_knn = best_knn.predict(X_test)
report_metrics(y_test, y_pred_best_knn, y_prob_best_knn, f"Best k-NN (k={best_k})")


# 6. Comparison: Logistic vs k-NN (same data, same splits)

Side-by-side **CV** scores (honest selection) and **test** confusion matrices (one final look).


In [ ]:
# --- Side-by-side CV summary table ---
compare_cv = pd.concat([
    log_cv.loc[log_cv["C"] == best_C, ["model", "param", "cv_f1_mean", "cv_f1_std", "cv_auc_mean", "cv_auc_std"]],
    knn_cv.loc[knn_cv["k"] == best_k, ["model", "param", "cv_f1_mean", "cv_f1_std", "cv_auc_mean", "cv_auc_std"]],
], ignore_index=True)

# Also show top of each grid for context
print("Best-by-CV models:")
display(compare_cv.round(3))

print("\nFull logistic C grid (top by F1):")
display(log_cv.sort_values("cv_f1_mean", ascending=False).head(3)[
    ["param", "cv_f1_mean", "cv_auc_mean"]
].round(3))
print("Full k-NN k grid (top by F1):")
display(knn_cv.sort_values("cv_f1_mean", ascending=False).head(3)[
    ["param", "cv_f1_mean", "cv_auc_mean"]
].round(3))

# --- Test metrics table ---
def row(name, y_pred, y_prob):
    return {
        "model": name,
        "test_acc": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_auc": roc_auc_score(y_test, y_prob),
    }

test_table = pd.DataFrame([
    row(f"Logistic C={best_C}", y_pred_best_log, y_prob_best_log),
    row(f"k-NN k={best_k}", y_pred_best_knn, y_prob_best_knn),
])
print("\nHeld-out TEST metrics (after CV selection):")
display(test_table.round(3))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best_log, display_labels=["stay", "churn"], ax=axes[0]
)
axes[0].set_title(f"Logistic C={best_C}")
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best_knn, display_labels=["stay", "churn"], ax=axes[1]
)
axes[1].set_title(f"k-NN k={best_k}")
plt.suptitle("Test confusion matrices (threshold / vote ≈ 0.5)")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4.5))
RocCurveDisplay.from_predictions(y_test, y_prob_best_log, name=f"Logistic C={best_C}", ax=ax)
RocCurveDisplay.from_predictions(y_test, y_prob_best_knn, name=f"k-NN k={best_k}", ax=ax)
ax.plot([0, 1], [0, 1], "k--", lw=1, label="chance")
ax.set_title("ROC on test")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


## When to prefer which? (say out loud in class)

| | **Logistic regression** | **k-NN** |
|---|---|---|
| Decision shape | Global **linear** boundary (in feature / log-odds space) | **Local** neighbourhood vote |
| Hyperparameter | `C` (regularization) | `k` (neighbour count) |
| Scaling | Helpful (esp. with regularization) | **Critical** — distances dominate |
| Interpretability | Coefficients tell a story | Lazy memory — hard to explain globally |
| Data regime | Works well if signal is roughly linear | Can win when decision regions are **locally irregular** |
| Cost at predict time | Cheap (dot product) | Heavier (search neighbours) |
| Bias–variance | Small `C` ↑ bias; large `C` ↑ variance | Small `k` ↑ variance; large `k` ↑ bias |

**Shipping takeaway:** try both with the **same** CV protocol; pick with the metric that matches FP/FN costs — not “accuracy vibes.”


### TODO 3 (students) — fill the comparison claim

Complete the sentence with evidence from the tables above:

> On this churn dataset, I would ship **__________** first because CV F1/AUC is __________ and the business cares most about __________.


In [ ]:
# TODO 3 — write your one-liner as a Python string (no wrong answer if grounded in the table)
my_choice = "Logistic"  # or "k-NN"
reason = "..."          # cite cv_f1 / recall / interpretability
print(f"Ship: {my_choice}. Reason: {reason}")


# 7. Threshold tuning for the business objective

Assume a simple cost model for the **retention** campaign:

- Cost of **FN** (missed churner) = 5  
- Cost of **FP** (unnecessary offer) = 1  

We sweep thresholds on **train** predictions (or a validation fold). Do **not** pick the threshold by maximizing test metrics.


In [ ]:
# Use best logistic probabilities on TRAIN to choose threshold (honest)
train_prob = best_log.predict_proba(X_train)[:, 1]

COST_FN, COST_FP = 5.0, 1.0
thresholds = np.linspace(0.05, 0.95, 37)
rows = []
for t in thresholds:
    pred = (train_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_train, pred).ravel()
    cost = COST_FN * fn + COST_FP * fp
    rows.append({
        "threshold": t,
        "precision": precision_score(y_train, pred, zero_division=0),
        "recall": recall_score(y_train, pred, zero_division=0),
        "f1": f1_score(y_train, pred, zero_division=0),
        "cost": cost,
    })
thr_df = pd.DataFrame(rows)
best_t = thr_df.loc[thr_df["cost"].idxmin(), "threshold"]
print(f"Best threshold on TRAIN by cost: {best_t:.2f}")
display(thr_df.loc[thr_df["cost"].idxmin()].to_frame().T.round(3))

plt.figure(figsize=(7, 3.5))
plt.plot(thr_df["threshold"], thr_df["cost"], label="expected cost proxy")
plt.axvline(best_t, color="k", ls="--", label=f"best t={best_t:.2f}")
plt.xlabel("threshold")
plt.ylabel(f"cost = {COST_FN}*FN + {COST_FP}*FP")
plt.title("Threshold sweep on TRAIN (logistic)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Apply chosen threshold once on TEST
y_pred_tuned = (y_prob_best_log >= best_t).astype(int)
print("\nTEST metrics at default 0.5 vs tuned threshold:")
print("thr=0.50", "F1", round(f1_score(y_test, y_pred_best_log), 3),
      "Recall", round(recall_score(y_test, y_pred_best_log), 3))
print(f"thr={best_t:.2f}", "F1", round(f1_score(y_test, y_pred_tuned), 3),
      "Recall", round(recall_score(y_test, y_pred_tuned), 3))


### TODO 4 (students) — change the cost ratio

Set `COST_FN, COST_FP = 2, 1` (or `10, 1`) and re-run the sweep mentally: does the best threshold move **down** (more aggressive churn flags) when FN gets more expensive?


In [ ]:
# TODO 4 — quick check: print best thresholds for two cost ratios
for COST_FN, COST_FP in [(2, 1), (5, 1), (10, 1)]:
    best = None
    best_cost = np.inf
    for t in thresholds:
        pred = (train_prob >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_train, pred).ravel()
        cost = COST_FN * fn + COST_FP * fp
        if cost < best_cost:
            best_cost, best = cost, t
    print(f"FN:{COST_FN} FP:{COST_FP}  -> best thr≈{best:.2f}  train_cost={best_cost:.0f}")


# 8. What we’d do in production (checklist)

1. **Freeze** feature definitions + training window; document label delay (churn realized next cycle).
2. **Pipeline** in one object: impute/encode/scale + model (here: scaler + logistic or k-NN).
3. **Monitor** weekly: churn rate, prediction rate, precision/recall at the **shipped** threshold.
4. Watch for **data drift**: feature distributions shift (pricing change, new plan mix) → CV scores from last month may no longer hold.
5. Re-train / re-tune `C` or `k` and threshold on a schedule; keep a **holdout** or time-based backtest.
6. Prefer **interpretable** logistic for stakeholder buy-in unless k-NN clearly wins on the product metric.
7. Still no trees/boosting today — but the **evaluation discipline** (CV, metrics, threshold) stays the same when you add them later.


# Wrap-up

1. Churn is a **binary** industry problem; FP vs FN costs drive the metric and threshold.
2. **Logistic:** scale → fit → `predict_proba` → metrics; tune **`C`** with CV; coefficients explain.
3. **k-NN:** scale (**must**) → choose **`k`** with CV → vote; small `k` = high variance.
4. **Compare** on the same splits with a CV F1/AUC table — then one test look.
5. **Threshold** is a product knob; tune on train/validation, not by peeking at test.
